# JurisTrace S4 Unified — SFT Qwen3.8-27B no Colab (G4 96GB / H100 / A100)

Treinamento unificado **S4 (`50.150` trajetórias: `30.101` S2 Desengessado + `19.640` S4 Processo Legislativo Nativo Câmara/Senado A–F + `409` Gold Enriquecimento Editorial/CoT)** a partir do base limpo `Qwen/Qwen3.8-27B` com **Unsloth QLoRA 4-bit + MTP head nativo (`mtp.*`) em BF16** (CE auxiliar `0,3`) e mascaramento de `<think>` vazio em turnos intermediários de `tool_calls`.

**Zero Upload Manual:**
- Ao dar **`Ctrl+F9` (Executar tudo)**, a Célula 2 baixa automaticamente o dataset criptografado (`SFT_TRAIN_S4_UNIFIED.jsonl.gz.enc`, 218 MB) e o payload (`colab_s4_payload_v1.zip.enc`), valida o SHA-256, espelha no seu Google Drive e inicia o treino.
- Checkpoints atômicos (`adapter_model.safetensors` + `mtp.safetensors` + estado do otimizador) são espelhados para `/content/drive/MyDrive/JURISTRACE_COLAB/runs/s4_sft_v1` a cada **50 steps**.
- Se a sessão cair e você reconectar e apertar `Ctrl+F9`, a Célula 7 retoma sozinha do último checkpoint completo (`--resume AUTO`).
- Ao concluir 100%, a Célula 8 desconecta a GPU automaticamente (`runtime.unassign()`) para economizar CCUs.


In [ ]:
# 1) GPU — confirme que conectou em G4 (96 GB) / H100 (80 GB) / A100
!nvidia-smi
import torch
p = torch.cuda.get_device_properties(0)
vram_gb = p.total_memory / 1e9
PROFILE = "v44_s4_8k" if vram_gb >= 70 else "v42_core_4k"
print(f"{p.name} | {vram_gb:.1f} GB VRAM | CC {p.major}.{p.minor} | PROFILE={PROFILE}")
assert p.total_memory > 35e9, "Runtime sem GPU premium — selecione G4 / H100 / A100 em Tempo de execução > Alterar tipo."
!df -h /content

In [ ]:
# 2) Drive + Auto-Bootstrap Criptografado S4 (Zero Upload Manual)
import hashlib, os, shutil, subprocess, zipfile
from google.colab import drive

drive.mount("/content/drive")

CANDIDATE_ROOTS = [
    "/content/drive/MyDrive/JURISTRACE_COLAB",
    "/content/drive/MyDrive/juristrace_s4",
]
DRIVE_ROOT = next((p for p in CANDIDATE_ROOTS if os.path.isdir(p)), CANDIDATE_ROOTS[0])
os.makedirs(f"{DRIVE_ROOT}/data", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/code", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/runs/s4_sft_v1", exist_ok=True)

DATASET = "/content/SFT_TRAIN_S4_UNIFIED.jsonl"
DRIVE_ENC_CACHE = f"{DRIVE_ROOT}/data/SFT_TRAIN_S4_UNIFIED.jsonl.gz.enc"
PAYLOAD = f"{DRIVE_ROOT}/code/colab_s4_payload_v1.zip"
EXPECTED_DATASET_SHA256 = "c7505cd53e506a930c600ae43aaa9556c422c64adfeb7564cf3c9cd7be8aa1e9"
EXPECTED_PAYLOAD_SHA256 = "46af1f15029785ad26cec5e7a9dc1085606e99590779c6e12d6b48ae9ce04f7d"
AES_KEY = "f698445148d8dc8af6d3b960c23bd6c57d181790b693f092428ff5e7dfa188a7"
BASE_REL = "https://github.com/guilhermedemelocabral/juristrace-s2-colab-bootstrap/releases/download/v4.0-s4-enc"

def sha256(path, buf=1 << 20):
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        while chunk := fh.read(buf):
            h.update(chunk)
    return h.hexdigest()

if not os.path.isfile(DATASET) or sha256(DATASET) != EXPECTED_DATASET_SHA256:
    enc_src = DRIVE_ENC_CACHE if (os.path.isfile(DRIVE_ENC_CACHE) and os.path.getsize(DRIVE_ENC_CACHE) > 200_000_000) else "/content/SFT_TRAIN_S4_UNIFIED.jsonl.gz.enc"
    if enc_src == "/content/SFT_TRAIN_S4_UNIFIED.jsonl.gz.enc":
        print("Baixando SFT_TRAIN_S4_UNIFIED.jsonl.gz.enc (218 MB) em velocidade máxima...")
        subprocess.check_call(["curl", "-fSL", f"{BASE_REL}/SFT_TRAIN_S4_UNIFIED.jsonl.gz.enc", "-o", enc_src])
        shutil.copy2(enc_src, DRIVE_ENC_CACHE)
    print("Descriptografando e descomprimindo SFT_TRAIN_S4_UNIFIED.jsonl (915 MB, 50.150 trajetórias) no NVMe local...")
    subprocess.check_call(
        f"openssl enc -d -aes-256-cbc -pbkdf2 -pass pass:{AES_KEY} -in '{enc_src}' | gzip -dc > '{DATASET}'",
        shell=True,
    )
    if os.path.isfile("/content/SFT_TRAIN_S4_UNIFIED.jsonl.gz.enc"):
        os.remove("/content/SFT_TRAIN_S4_UNIFIED.jsonl.gz.enc")

if not os.path.isfile(PAYLOAD) or sha256(PAYLOAD) != EXPECTED_PAYLOAD_SHA256:
    print("Baixando e descriptografando colab_s4_payload_v1.zip direto para o Google Drive...")
    subprocess.check_call(["curl", "-fSL", f"{BASE_REL}/colab_s4_payload_v1.zip.enc", "-o", "/content/colab_s4_payload_v1.zip.enc"])
    subprocess.check_call(["openssl", "enc", "-d", "-aes-256-cbc", "-pbkdf2", "-pass", f"pass:{AES_KEY}", "-in", "/content/colab_s4_payload_v1.zip.enc", "-out", PAYLOAD])
    os.remove("/content/colab_s4_payload_v1.zip.enc")

ds = sha256(DATASET); print("dataset sha256:", ds)
assert ds == EXPECTED_DATASET_SHA256, "SHA do dataset S4 difere do selado — NÃO treine; confira o corte."
pl = sha256(PAYLOAD); print("payload sha256:", pl)
assert pl == EXPECTED_PAYLOAD_SHA256, "Payload de código S4 difere do selado."
shutil.rmtree("/content/payload", ignore_errors=True)
os.makedirs("/content/payload", exist_ok=True)
with zipfile.ZipFile(PAYLOAD) as zf:
    zf.extractall("/content/payload")
print("payload S4 extraído em /content/payload | Drive pronto em:", DRIVE_ROOT)


In [ ]:
# 3) Stack — mesma pinagem validada do S1/S2 (sft-qwen38)
%pip install -q "unsloth" "transformers>=5.8.0" "datasets" "trl" "accelerate" "bitsandbytes" "peft" "safetensors"
import unsloth, transformers, torch, trl, peft, bitsandbytes  # noqa
from transformers.models.qwen3_5.modeling_qwen3_5 import Qwen3_5ForConditionalGeneration  # noqa
print("stack ok | transformers", transformers.__version__, "| torch", torch.__version__)

In [ ]:
# 4) Base — Qwen/Qwen3.8-27B na revisão pinada (54 GB, ~8–15 min no G4/H100)
from huggingface_hub import snapshot_download
import os
PIN = "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0"
MODEL_DIR = snapshot_download("Qwen/Qwen3.8-27B", revision=PIN, local_dir="/content/base", max_workers=8)
n = len([f for f in os.listdir(MODEL_DIR) if f.endswith(".safetensors")])
assert n == 18, f"shards incompletos: {n}/18"
print("base completa:", MODEL_DIR)

In [ ]:
# 5) Plano em seco — registra a receita exata do S4 no log (não treina)
import subprocess, glob, os
os.makedirs(f"{DRIVE_ROOT}/logs", exist_ok=True)
cmd = ["python", "scripts/train_s1_qwen38_27b.py",
       "--model-dir", "/content/base",
       "--train", DATASET,
       "--profile", PROFILE,
       "--out", "/content/runs/s4_sft_v1",
       "--save-steps", "50"]
plan = subprocess.run(cmd, cwd="/content/payload", capture_output=True, text=True)
print(plan.stdout)
if plan.returncode != 0:
    print(plan.stderr)
    raise SystemExit(plan.returncode)
stamp = __import__("time").strftime("%Y%m%d_%H%M%S")
with open(f"{DRIVE_ROOT}/logs/plan_s4_{stamp}.json", "w") as fh:
    fh.write(plan.stdout)
print("plano salvo em", f"{DRIVE_ROOT}/logs/plan_s4_{stamp}.json")


In [ ]:
# 6) SMOKE (Mantido RUN_SMOKE = False para ir direto ao treino completo no Ctrl+F9)
RUN_SMOKE = False
if RUN_SMOKE:
    import subprocess
    cmd = ["python", "scripts/train_s1_qwen38_27b.py", "--probe", "--go", "--max-steps", "20",
           "--model-dir", "/content/base",
           "--train", DATASET,
           "--profile", PROFILE,
           "--out", "/content/probe_s4"]
    rc = subprocess.call(cmd, cwd="/content/payload")
    if rc != 0:
        raise SystemExit(rc)
    print("SMOKE 20 steps OK!")
else:
    print("Pulando direto para o RUN COMPLETO S4 (Célula 7)...")


In [ ]:
# 7) RUN COMPLETO S4 COM AUTO-RESUME E CHECKPOINTS ATÔMICOS A CADA 50 STEPS
# Se já houver checkpoint completo no Drive, retoma automaticamente (--resume AUTO).
import os, shutil, subprocess, threading, time

SRC = "/content/runs/s4_sft_v1"
DST = f"{DRIVE_ROOT}/runs/s4_sft_v1"
TRAIN = DATASET
os.makedirs(SRC, exist_ok=True)
os.makedirs(DST, exist_ok=True)

REQUIRED_FILES = ("mtp.safetensors", "adapter_model.safetensors", "adapter_config.json", "trainer_state.json", "optimizer.pt", "scheduler.pt")

def is_ckpt_complete(path):
    return os.path.isdir(path) and all(os.path.isfile(os.path.join(path, f)) and os.path.getsize(os.path.join(path, f)) > 0 for f in REQUIRED_FILES)

# Busca automática na pasta compartilhada (sharedWithMe) via Drive API v3 caso não haja atalho no Meu Drive
def pull_latest_from_shared_drive(run_name, src_dir, req_files):
    try:
        import glob
        for sc in glob.glob(f"/content/drive/.shortcut-targets-by-id/*/JURISTRACE_COLAB/runs/{run_name}"):
            if os.path.isdir(sc):
                return sc
        from google.colab import auth
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        auth.authenticate_user()
        srv = build("drive", "v3", cache_discovery=False)
        q_root = "name = 'JURISTRACE_COLAB' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
        roots = srv.files().list(q=q_root, fields="files(id,name,shared)", includeItemsFromAllDrives=True, supportsAllDrives=True).execute().get("files", [])
        candidates = []
        for r in roots:
            q_runs = f"'{r['id']}' in parents and name = 'runs' and trashed = false"
            runs_f = srv.files().list(q=q_runs, fields="files(id)", includeItemsFromAllDrives=True, supportsAllDrives=True).execute().get("files", [])
            for rf in runs_f:
                q_sub = f"'{rf['id']}' in parents and name = '{run_name}' and trashed = false"
                sub_f = srv.files().list(q=q_sub, fields="files(id)", includeItemsFromAllDrives=True, supportsAllDrives=True).execute().get("files", [])
                for sf in sub_f:
                    q_ck = f"'{sf['id']}' in parents and name contains 'checkpoint-' and trashed = false"
                    ckpts = srv.files().list(q=q_ck, fields="files(id,name)", pageSize=200, includeItemsFromAllDrives=True, supportsAllDrives=True).execute().get("files", [])
                    for c in ckpts:
                        if c["name"].startswith("checkpoint-") and c["name"].split("-")[1].isdigit():
                            candidates.append((int(c["name"].split("-")[1]), c))
        candidates.sort(key=lambda x: x[0], reverse=True)
        for step_num, ck in candidates:
            q_files = f"'{ck['id']}' in parents and trashed = false"
            flist = {f["name"]: f for f in srv.files().list(q=q_files, fields="files(id,name,size)", includeItemsFromAllDrives=True, supportsAllDrives=True).execute().get("files", [])}
            if all(k in flist and int(flist[k].get("size", 0)) > 0 for k in req_files):
                target_ck = os.path.join(src_dir, ck["name"])
                if not is_ckpt_complete(target_ck):
                    print(f"[SHARED-DRIVE] Baixando {ck['name']} (step {step_num}) da pasta compartilhada JURISTRACE_COLAB...")
                    os.makedirs(target_ck, exist_ok=True)
                    for fname, fmeta in flist.items():
                        if int(fmeta.get("size", 0)) > 0:
                            req = srv.files().get_media(fileId=fmeta["id"])
                            with open(os.path.join(target_ck, fname), "wb") as fh:
                                dl = MediaIoBaseDownload(fh, req, chunksize=32*1024*1024)
                                done = False
                                while not done:
                                    _, done = dl.next_chunk()
                return src_dir
    except Exception as e:
        print("[SHARED-DRIVE INFO]", e)
    return None
shared_dir = pull_latest_from_shared_drive("s4_sft_v1", SRC, REQUIRED_FILES)
if shared_dir and shared_dir != SRC:
    DST = shared_dir
drive_ckpts = sorted(
    set(d for d in (os.listdir(DST) + os.listdir(SRC)) if d.startswith("checkpoint-") and (is_ckpt_complete(os.path.join(DST, d)) or is_ckpt_complete(os.path.join(SRC, d)))),
    key=lambda s: int(s.split("-")[1])
)
_old_drive_ckpts = sorted(
    (d for d in os.listdir(DST) if d.startswith("checkpoint-") and is_ckpt_complete(os.path.join(DST, d))),
    key=lambda s: int(s.split("-")[1])
)
resume_args = []
if drive_ckpts:
    latest = drive_ckpts[-1]
    local_latest = os.path.join(SRC, latest)
    if not is_ckpt_complete(local_latest):
        print(f"[AUTO-RESUME] Restaurando {latest} do Drive para /content...")
        shutil.rmtree(local_latest, ignore_errors=True)
        shutil.copytree(os.path.join(DST, latest), local_latest) if is_ckpt_complete(os.path.join(DST, latest)) else None
    print(f"[AUTO-RESUME] Retomando treinamento S4 a partir de {latest} (--resume AUTO)")
    resume_args = ["--resume", "AUTO"]
else:
    print("[START] Nenhum checkpoint S4 prévio no Drive; iniciando do step 0.")

def mirror_once(src, dst):
    if not os.path.isdir(src):
        return
    for name in sorted(os.listdir(src)):
        s, d = os.path.join(src, name), os.path.join(dst, name)
        if name.startswith("checkpoint-"):
            if is_ckpt_complete(s) and os.path.isfile(os.path.join(s, "CHECKPOINT_READY.json")) and not is_ckpt_complete(d):
                tmp_d = d + ".tmp_copy"
                print(f"[MIRROR] Espelhando {name} atomicamente para o Drive...", flush=True)
                shutil.rmtree(tmp_d, ignore_errors=True)
                shutil.copytree(s, tmp_d)
                shutil.rmtree(d, ignore_errors=True)
                os.rename(tmp_d, d)
                print(f"[MIRROR] {name} salvo e garantido no Drive!", flush=True)
        elif name in ("adapter_model.safetensors", "adapter_config.json", "mtp.safetensors", "tokenizer.json", "tokenizer_config.json", "chat_template.jinja"):
            if os.path.isfile(s) and (not os.path.isfile(d) or os.path.getsize(s) != os.path.getsize(d)):
                tmp_f = d + ".tmp"
                shutil.copy2(s, tmp_f)
                os.replace(tmp_f, d)
    # Mantém no Drive os 4 checkpoints completos mais recentes + marcos a cada 1000 steps
    complete_dst = sorted(
        (c for c in os.listdir(dst) if c.startswith("checkpoint-") and is_ckpt_complete(os.path.join(dst, c))),
        key=lambda x: int(x.split("-")[1])
    )
    if len(complete_dst) > 4:
        for old_c in complete_dst[:-4]:
            if int(old_c.split("-")[1]) % 1000 != 0:
                shutil.rmtree(os.path.join(dst, old_c), ignore_errors=True)

def mirror_loop(src, dst, stop):
    while not stop.is_set():
        try:
            mirror_once(src, dst)
        except Exception as e:
            print("[MIRROR WARN]", e, flush=True)
        time.sleep(30)

stop = threading.Event()
t = threading.Thread(target=mirror_loop, args=(SRC, DST, stop), daemon=True)
t.start()
if drive_ckpts:
    ts_path = os.path.join(SRC, drive_ckpts[-1], "trainer_state.json")
    if os.path.isfile(ts_path):
        ts_data = json.load(open(ts_path))
        print(f"[CHECKPOINT INFO] global_step={ts_data.get('global_step')} | max_steps={ts_data.get('max_steps')} | epoch={ts_data.get('epoch'):.4f}", flush=True)

env = os.environ.copy()
env["PYTHONUNBUFFERED"] = "1"
env["TORCHINDUCTOR_COMPILE_THREADS"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"

cmd = ["python", "-u", "scripts/train_s1_qwen38_27b.py", "--go",
       "--model-dir", "/content/base",
       "--train", TRAIN,
       "--profile", PROFILE,
       "--out", SRC, "--save-steps", "50"] + resume_args

print("[TRAIN] Iniciando processo com TORCHINDUCTOR_COMPILE_THREADS=1 e log ao vivo...", flush=True)
proc = subprocess.Popen(cmd, cwd="/content/payload", env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="", flush=True)
rc = proc.wait()
stop.set()
mirror_once(SRC, DST)
if rc != 0:
    raise SystemExit(rc)
print("TREINO COMPLETO S4 CONCLUÍDO COM SUCESSO! Todos os checkpoints estão salvos no Drive.")

In [ ]:
# 8) Entrega + Desconexão Automática da GPU (para não gastar CCU após terminar)
import os
DST = f"{DRIVE_ROOT}/runs/s4_sft_v1"
for name in sorted(os.listdir(DST)):
    p = os.path.join(DST, name)
    size = sum(os.path.getsize(os.path.join(p, f)) for f in os.listdir(p)) if os.path.isdir(p) else os.path.getsize(p)
    print(f"{size/1e6:9.1f} MB  {name}")
print("\nTreino S4 100% salvo no Google Drive! Desconectando a GPU automaticamente para preservar seus CCUs...")
from google.colab import runtime
runtime.unassign()
